# SKKB — Baseline Judge Run (local / Databricks dual-mode)

JIRA: **GCAI-3092** — diagnostic LLM-as-a-Judge evaluation for the SK Knowledge-Base chatbot.

This notebook runs the `hg_ds_evals` judge against the SKKB input table.

- **YAML config:** [`configs/skkb/skkb_exp_001_baseline.yaml`](../../configs/skkb/skkb_exp_001_baseline.yaml)
- **System template:** [`configs/skkb/system.md.j2`](../../configs/skkb/system.md.j2)
- **Judge:** `gpt-5-1` via `databricks_async`, `reasoning_effort: high`
- **Output:** CSV checkpoint under `checkpoints/databricks_gpt51/` (local working directory)

## Run modes
- `RUN_ON_DBX = True` — runs on a Databricks cluster, reads the UC table via `spark.read.table(...)`.
- `RUN_ON_DBX = False` — runs on your laptop, reads the UC table via `databricks-sql-connector` against a SQL warehouse, authenticates with the named CLI profile, writes results to the local `checkpoints/` directory.

> **NOTE — why we do not call `run_experiment` directly:** the upstream
> `run_experiment` helper instantiates `PromptBuilder(rubric=rubric)`
> without passing the YAML's `paths.system_template_path`, so the
> default embedded template is used and our `critical_evaluation_rules`,
> `domain_specific_guidance`, `final_reminders`, and the baked-in
> **AGENT CONTEXT** are silently dropped. To keep the custom template
> in effect we replicate `run_experiment`'s flow explicitly below.

Run 
`databricks auth login --host https://adb-3174992876438447.7.azuredatabricks.net` to prevent the auth errors.

## Run mode

In [1]:
# Toggle to run on a Databricks cluster vs. locally from your laptop.
RUN_ON_DBX: bool = False

# Local-only: which Databricks CLI profile to use for auth.
DBX_PROFILE: str = "adb-chat"

DBX_CLUSTER_ID: str | None = "0424-122149-m97focly"
DBX_SQL_WAREHOUSE_ID: str | None = None

## Environment & imports

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
if RUN_ON_DBX:
    # Cluster-only bootstrap. Skipped locally.
    !pip install -qe /Workspace/Users/sg7cb@s-mxs.net/hey-george-ds/ds_common
    !pip install openai -q
    dbutils.library.restartPython()  # noqa: F821 (provided by DBR)
else:
    print("Local mode — skipping cluster pip installs / restartPython.")
    print("Install once into your venv if missing:")
    print("    uv pip install --system-certs databricks-sql-connector openai")

Local mode — skipping cluster pip installs / restartPython.
Install once into your venv if missing:
    uv pip install --system-certs databricks-sql-connector openai


In [4]:
import os
import sys
from pathlib import Path

# Make the local hg-ds-evals repo importable when running off-cluster.
if not RUN_ON_DBX:
    _repo_root = Path.cwd()
    while _repo_root != _repo_root.parent and not (_repo_root / "hg_ds_evals").is_dir():
        _repo_root = _repo_root.parent
    if (_repo_root / "hg_ds_evals").is_dir() and str(_repo_root) not in sys.path:
        sys.path.insert(0, str(_repo_root))
        print(f'Local repo root added to sys.path: "{_repo_root}"')

    os.environ["DATABRICKS_CONFIG_PROFILE"] = DBX_PROFILE

    from databricks.sdk import WorkspaceClient
    _w = WorkspaceClient()
    print(f"Authenticated as: {_w.current_user.me().user_name}")
    print(f"Workspace host:   {_w.config.host}")

import config_nbs
config_nbs.add_local_paths()

import hg_ds_evals
print(hg_ds_evals.__file__)

import pandas as pd
pd.set_option("display.width", None)

Local repo root added to sys.path: "/Users/SG7CB/Developer/hg-ds-evals"
Authenticated as: sg7cb@s-mxs.net
Workspace host:   https://adb-3174992876438447.7.azuredatabricks.net
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/" added!
Path "/Workspace/Users/sg7cb@s-mxs.net/hg-ds-evals/experiments/skkb/" added!
/Users/SG7CB/Developer/hg-ds-evals/hg_ds_evals/__init__.py


In [5]:
from databricks.sdk import WorkspaceClient
for w in WorkspaceClient().warehouses.list():
    print(w.id, w.state, w.name)

eab78c481da6edd8 State.STOPPED Serverless Starter Warehouse


## UC table reader

Local: `databricks-sql-connector` against a SQL warehouse, authenticating via the same CLI profile.
Cluster: `spark.read.table(...)`.

In [6]:
def _resolve_http_path(workspace_client, *, cluster_id=None, warehouse_id=None) -> tuple[str, str]:
    host = workspace_client.config.host.replace("https://", "").rstrip("/")
    if cluster_id:
        # All-purpose cluster path: /sql/protocolv1/o/<org_id>/<cluster_id>
        org_id = host.split(".")[0].removeprefix("adb-").split(".")[0]
        # Better: read it from the workspace URL: adb-<org_id>.<n>.azuredatabricks.net
        org_id = host.split("adb-")[1].split(".")[0]
        return host, f"/sql/protocolv1/o/{org_id}/{cluster_id}"
    # Warehouse fallback (existing behavior)
    warehouses = list(workspace_client.warehouses.list())
    match = next((w for w in warehouses if w.id == warehouse_id), None) if warehouse_id else None
    if match is None:
        running = [w for w in warehouses if str(w.state) == "RUNNING"]
        match = (running or warehouses)[0]
    return host, f"/sql/1.0/warehouses/{match.id}"


def read_uc_table(table_fqn: str) -> pd.DataFrame:
    """Read a Unity Catalog table into pandas — works locally (SQL warehouse) and on Databricks (Spark)."""
    if RUN_ON_DBX:
        return spark.read.table(table_fqn).toPandas()  # noqa: F821 (provided by DBR)

    from databricks import sql as dbx_sql
    server_hostname, http_path = _resolve_http_path(
        _w, cluster_id=DBX_CLUSTER_ID, warehouse_id=DBX_SQL_WAREHOUSE_ID
        )
    # `access_token` callable refreshes OAuth U2M tokens transparently.
    access_token = _w.config.authenticate()["Authorization"].removeprefix("Bearer ")
    with dbx_sql.connect(
        server_hostname=server_hostname,
        http_path=http_path,
        access_token=access_token,
    ) as conn:
        with conn.cursor() as cur:
            cur.execute(f"SELECT * FROM {table_fqn}")
            return cur.fetchall_arrow().to_pandas()

## Load the experiment YAML and build the rubric

In [7]:
YAML_FILE_NAME = "skkb_exp_001_baseline_no_expected_enums"
# Custom suffix appended to the checkpoint filename so a new run does not
# overwrite a prior checkpoint. Lands as `..._<reasoning>_<CHECKPOINT_SUFFIX>[_test].csv`.
# Set to "" to keep the old behaviour. Example: "2026_05_01_score".
CHECKPOINT_SUFFIX = "online_nightly_2026_05_01_score"

# Databricks CLI profile (~/.databrickscfg) used both for the model-serving
# call and for unattended OAuth token refresh during long runs. Empty string
# defers to the default profile or DATABRICKS_CONFIG_PROFILE env var.
# IMPORTANT: run `databricks auth login --profile <name>` once before the
# run to seed the OAuth refresh token; subsequent token rotation is then
# automatic and does not require browser interaction.
DATABRICKS_PROFILE = ""

# When True, drop existing `error=True` rows from the checkpoint file
# before the run starts (with timestamped backup). Lets a re-run cleanly
# retry test cases that failed for transient reasons (e.g. token expiry)
# without accumulating duplicate error rows over retry cycles.
PURGE_ERRORS_BEFORE_RUN = True
config_path = f"../configs/skkb/{YAML_FILE_NAME}.yaml"
assert Path(config_path).exists(), f"config file not found: {config_path}"

In [8]:
from hg_ds_evals.rubrics.loader import (
    load_experiment_config,
    build_rubric_from_config,
    get_experiment_name,
)

config = load_experiment_config(config_path)
rubric = build_rubric_from_config(config)

print(f"Experiment       : {config['experiment']['name']}")
print(f"Rubric name      : {rubric.metadata.name}")
print(f"Rubric persona   : {rubric.metadata.persona}")
print(f"Dimensions       : {rubric.dimension_ids}")
print(f"Input fields     : {rubric.input_field_names}")
print(f"Output fields    : {[f.name for f in rubric.output_schema.fields]}")
print(f"Pass threshold   : {rubric.pass_threshold}")
print(f"Judge model      : {config['model']['model_deployment_name']} "
      f"({config['model']['api_provider']}, reasoning={config['model']['reasoning_effort']})")
print(f"Input table      : {config['dataset']['input_schema']}.{config['dataset']['input_dataset']}")
print(f"test_num_rows    : {config['dataset']['test_num_rows']}")

Experiment       : skkb_exp_001_baseline_no_expected_enums
Rubric name      : Custom Rubric
Rubric persona   : You are an expert evaluator assessing the end-to-end quality of a Slovak retail-banking knowledge-base (KB) chatbot ('Hey George'), with the goal of pinpointing concrete improvements for the bot/agent team, the retrieval/reranker team, and the KB content team.

Dimensions       : ['query_clarity', 'selection_semantic_relevance', 'selected_context_sufficiency', 'optimal_retrieved_context_adequacy', 'answer_expected_alignment', 'answer_groundedness', 'language_compliance']
Input fields     : ['user_query', 'pre_prune_enum_ids', 'post_prune_candidates_text', 'post_prune_enum_ids', 'reranked_enum_ids', 'reranked_kb_context', 'expected_response', 'agent_response']
Output fields    : ['case_scope', 'expected_reference_looks_wrong', 'expected_reference_issue_description', 'optimal_enum_selection', 'expected_answer_summary_with_optimal_context', 'extra_or_distracting_enums', 'unavaila

## Load the input data

In [9]:
input_table = (
    f"{config['dataset']['input_catalog']}."
    f"{config['dataset']['input_schema']}."
    f"{config['dataset']['input_dataset']}"
)
print(input_table)

df_input = read_uc_table(input_table)
print(f"rows: {len(df_input):,}")
df_input.head()

prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_online_nightly_2026_05_01_score
rows: 513


,test_case_id,trace_id,request_time,execution_duration_ms,user_query,knowledge_search_run_count,knowledge_search_final_run_index,knowledge_search_runs,reranked_kb_context,kb_version,...,categories_list,reranked_enums_kb_sk,reranked_enums_kb_en,post_prune_candidates_kb_sk,post_prune_candidates_kb_en,user_query_en,expected_response_en,agent_response_en,execution_duration_s,execution_duration_min
0,Test case 512,tr-d64eae1ae2dfc2c0a4488cce8a9e2d14,1777657823605,38362,"Potrebujem niečo riešiť ohľadom účtu, môžem na...",0,0,[],,,...,"[""domain_knowledge"", ""routing_kb_only""]",[],[],[],[],I need to sort something out regarding my acco...,"I apologize, I do not have information availab...","Yes, you can also write directly to your bank ...",38.362,0.639367
1,Test case 511,tr-c88923478e40559fb4ed62efbe91c4cb,1777657816057,52393,Kde nájdem kontakt na svojho konzultanta?,1,1,"[{""run_index"": 1, ""knowledge_search_span_id"": ...",[KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2]\nWRI...,KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2,...,"[""domain_knowledge"", ""routing_kb_only""]","[{""enum_id"": ""WRITE_MESSAGE"", ""description"": ""...","[{""enum_id"": ""WRITE_MESSAGE"", ""description"": ""...","[{""enum_id"": ""WRITE_MESSAGE"", ""description"": ""...","[{""enum_id"": ""WRITE_MESSAGE"", ""description"": ""...",Where can I find the contact details for my co...,"I apologize, I did not find information regard...",You can find your consultant’s contact details...,52.393,0.873217
2,Test case 510,tr-ebdb8648b75ffbd4ba47196d8331dd72,1777657809845,63628,"Plánujem platiť mobilom a chcem vedieť, či sú ...",1,1,"[{""run_index"": 1, ""knowledge_search_span_id"": ...",[KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2]\nTOK...,KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2,...,"[""domain_knowledge"", ""routing_kb_only""]","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...",I plan to pay by mobile and I want to know whe...,Your card details are secure when paying by mo...,Your card details are very well protected when...,63.628,1.060467
3,Test case 508,tr-5904c234a23e502e3b7a8da17608f150,1777657800391,11342,Čo je tokenizácia platobnej karty?,0,0,[],,,...,"[""domain_knowledge"", ""routing_kb_only""]",[],[],[],[],What is payment card tokenization?,"I apologize, but I don’t have any information ...",Tokenization of a payment card is a security p...,11.342,0.189033
4,Test case 509,tr-1e8a5f527801cb71c966adb17a39387f,1777657800391,28613,Ako si tokenizujem kartu?,1,1,"[{""run_index"": 1, ""knowledge_search_span_id"": ...",[KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2]\nTOK...,KB_GAI_SK_SK_2026-04-20_14h16_phase_1_2,...,"[""domain_knowledge"", ""routing_kb_only""]","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...","[{""enum_id"": ""TOKENIZE_CARD"", ""description"": ""...",How do I tokenize my card?,"I’m sorry, but I don’t have information on how...",Tokenization is done automatically when you ad...,28.613,0.476883


## Build the judge's system + user prompts from the **custom** template

We explicitly pass `system_template_path` and `user_template_path` — without this the default embedded template is used and the AGENT CONTEXT / critical_evaluation_rules / final_reminders sections are silently dropped.

In [10]:
from hg_ds_evals.prompts.builder import PromptBuilder
from hg_ds_evals.prompts.common import display_prompt

system_template_path = Path(config["paths"]["system_template_path"])
user_template_path = Path(config["paths"]["user_template_path"])

if not system_template_path.is_absolute():
    system_template_path = (Path.cwd() / system_template_path).resolve()
if not user_template_path.is_absolute():
    user_template_path = (Path.cwd() / user_template_path).resolve()

assert system_template_path.exists(), f"system template missing: {system_template_path}"
assert user_template_path.exists(), f"user template missing: {user_template_path}"
print(f"system template : {system_template_path}")
print(f"user template   : {user_template_path}")

builder = PromptBuilder(
    rubric=rubric,
    system_template_path=system_template_path,
    user_template_path=user_template_path,
)
system_prompt = builder.build_system_prompt()
print(f"system prompt   : {len(system_prompt)} chars")

system template : /Users/SG7CB/Developer/hg-ds-evals/experiments/skkb/configs/skkb/system.md.j2
user template   : /Users/SG7CB/Developer/hg-ds-evals/experiments/skkb/configs/skkb/user.md.j2
system prompt   : 25427 chars


In [11]:
display_prompt(system_prompt, title="System Prompt (SKKB baseline — with AGENT CONTEXT)", font_size=12)

## Preview the user prompt on one real row

In [12]:
sample_row = df_input.head(1).iloc[0].to_dict()

user_prompt = builder.build_user_prompt(sample_row)
display_prompt(user_prompt, title=f"User Prompt — {sample_row.get('test_case_id')}", font_size=12)

## Run the evaluation

The eval results land in the **local** `checkpoints/` directory configured by the YAML (`paths.checkpoint_dir`). On a cluster this is the workspace-mounted notebook dir; locally it's relative to your CWD.

In [13]:
from hg_ds_evals.common.utils import (
    load_checkpoint,
    filter_df_with_checkpoints,
    prepare_eval_sample,
)
from hg_ds_evals.llm.api_client import get_api_client
from hg_ds_evals.evals.evaluator import async_run_evals

experiment_name = get_experiment_name(config_path)
INPUT_CATALOG = config["dataset"]["input_catalog"]
INPUT_SCHEMA = config["dataset"]["input_schema"]
INPUT_TABLE = config["dataset"]["input_dataset"]
ID_COLUMNS = config["dataset"].get("id_columns", [])
NUM_ROWS = config["dataset"]["test_num_rows"]
CHECKPOINT_DIR = config["paths"].get("checkpoint_dir", "checkpoints")

print(f"[1/5] loading input table {INPUT_CATALOG}.{INPUT_SCHEMA}.{INPUT_TABLE}")
df = read_uc_table(f"{INPUT_CATALOG}.{INPUT_SCHEMA}.{INPUT_TABLE}")
print(f"      rows: {len(df):,}")

print("\n[2/5] preparing eval sample")
df_sample, file_name_eval = prepare_eval_sample(
    df=df,
    evals_name=experiment_name,
    reasoning_effort=config["model"]["reasoning_effort"],
    suffix=CHECKPOINT_SUFFIX or None,
    num_rows=NUM_ROWS,
)
cols = config["dataset"]["eval_columns"]
if ID_COLUMNS:
    cols = list(dict.fromkeys(ID_COLUMNS + cols))
passthrough = config["dataset"].get("passthrough_columns", [])
if passthrough:
    cols = list(dict.fromkeys(cols + passthrough))
# Tolerate columns that are missing locally (e.g. derived columns produced only by Spark notebooks).
missing_cols = [c for c in cols if c not in df_sample.columns]
if missing_cols:
    print(f"      WARN — missing columns dropped: {missing_cols}")
    cols = [c for c in cols if c in df_sample.columns]
df_eval = df_sample[cols].copy()
print(f"      eval df: {len(df_eval)} rows, {len(cols)} cols  checkpoint_file={file_name_eval}")

print("\n[3/5] loading checkpoint")
ckp_df, ckp_path = load_checkpoint(
    checkpoint_file_name=file_name_eval,
    checkpoint_dir=CHECKPOINT_DIR,
    purge_error_rows=PURGE_ERRORS_BEFORE_RUN,
)
df_eval = filter_df_with_checkpoints(df_eval, ckp_df, id_cols=ID_COLUMNS)
print(f"      {len(df_eval)} rows remain after filtering {len(ckp_df)} already-scored rows")
print(f"      checkpoint path: {ckp_path}")

print("\n[4/5] setting up API client")
client = get_api_client(
    model_deployment_name=config["model"]["model_deployment_name"],
    api_provider=config["model"]["api_provider"],
    databricks_endpoint_url=config["model"].get("databricks_endpoint_url"),
    databricks_base_url=config["model"].get("databricks_base_url"),
    databricks_workspace_host=config["model"].get("databricks_workspace_host"),
    databricks_profile=DATABRICKS_PROFILE or None,
)

print("\n[5/5] running evaluations")
results_df, metrics = await async_run_evals(
    df=df_eval,
    system_prompt=system_prompt,
    client=client,
    config=config,
    checkpoint_path=ckp_path,
    user_prompt_builder=builder.build_user_prompt,
)

print("\ndone.")
print(f"checkpoint  : {ckp_path}")
print(f"metrics     : {metrics}")

[1/5] loading input table prod_aut_chat00_lab_catalog.private_uc0115_aix_db.dts_eval_skkb_exp_001_online_nightly_2026_05_01_score
      rows: 513

[2/5] preparing eval sample
Sample size: 513
File name: evals_skkb_exp_001_baseline_no_expected_enums_high_online_nightly_2026_05_01_score.csv
      eval df: 513 rows, 51 cols  checkpoint_file=evals_skkb_exp_001_baseline_no_expected_enums_high_online_nightly_2026_05_01_score.csv

[3/5] loading checkpoint
🧹 Purged 358 error rows from checkpoint (kept 155); backup → evals_skkb_exp_001_baseline_no_expected_enums_high_online_nightly_2026_05_01_score.csv.bak.20260502_212755
✓ Loading existing checkpoint: checkpoints/databricks_gpt51/evals_skkb_exp_001_baseline_no_expected_enums_high_online_nightly_2026_05_01_score.csv
ℹ️ Filtered 155 rows from checkpoint
ℹ️ Remaining rows to process: 358
      358 rows remain after filtering 155 already-scored rows
      checkpoint path: checkpoints/databricks_gpt51/evals_skkb_exp_001_baseline_no_expected_enums_h

## Check the checkpoint

In [14]:
df_results = pd.read_csv(ckp_path)
print(f"checkpoint rows: {len(df_results)}")
print()
print("error counts:")
print(df_results.get("error", pd.Series([None]*len(df_results))).isna().value_counts(dropna=False))

dim_score_cols = [c for c in df_results.columns if c.endswith("_score")]
print()
print("dimension score summary:")
print(df_results[dim_score_cols].describe().round(2))

checkpoint rows: 513

error counts:
error
False    513
Name: count, dtype: int64

dimension score summary:
       expert_score  enum_relevance_score  query_clarity_score  \
count           0.0                513.00               513.00   
mean            NaN                  0.66                 1.74   
std             NaN                  0.42                 0.55   
min             NaN                  0.00                 0.00   
25%             NaN                  0.00                 2.00   
50%             NaN                  0.94                 2.00   
75%             NaN                  1.00                 2.00   
max             NaN                  1.00                 2.00   

       selection_semantic_relevance_score  selected_context_sufficiency_score  \
count                              513.00                              513.00   
mean                                 1.58                                1.61   
std                                  0.74              

## Next step

Open [`skkb_001_results_viewer.ipynb`](skkb_001_results_viewer.ipynb) to aggregate scores, compute the judge-vs-expert Spearman, split missed ENUMs into retriever-vs-reranker failures, and render the HTML report.